[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/05_ONNX_Operators_and_OpSets/01_Standard_Operators/Standard_Operators_Deep_Dive.ipynb)

# 5.1 Standard ONNX Operators — Deep Dive

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [What is an ONNX Operator?](#1-what-is-an-onnx-operator) | Formal definition, the (domain, op_type, version) triple |
| 2 | [Operator Categories](#2-operator-categories) | Taxonomy of 200+ standard ops |
| 3 | [Conv — Mathematical Definition](#3-conv) | Full convolution formula with grouped/depthwise |
| 4 | [MatMul and Gemm](#4-matmul-and-gemm) | Matrix multiplication semantics and fusion |
| 5 | [Softmax — Numerical Stability](#5-softmax) | Formula, log-sum-exp trick |
| 6 | [BatchNormalization](#6-batchnormalization) | Training vs inference formulas |
| 7 | [Activation Functions](#7-activation-functions) | Relu, Sigmoid, Tanh, Gelu and more |
| 8 | [Reduction Operators](#8-reduction-operators) | ReduceMean, ReduceSum, ArgMax |
| 9 | [Tensor Manipulation](#9-tensor-manipulation) | Reshape, Transpose, Gather, Slice |
| 10 | [Broadcasting Rules](#10-broadcasting-rules) | Formal definition of numpy-style broadcasting |
| 11 | [Building Complete Models](#11-building-complete-models) | Composing operators with ReferenceEvaluator |
| 12 | [Attention Mechanism Pattern](#12-attention-mechanism-pattern) | Multi-head attention from ONNX ops |
| 13 | [Operator Catalog Visualization](#13-operator-catalog-visualization) | Charts and statistics |
| 14 | [Key Takeaways](#14-key-takeaways) | Summary |

In [ ]:
# !pip install onnx numpy matplotlib --quiet

import numpy as np
import onnx
from onnx import helper, TensorProto, checker, numpy_helper, defs
from onnx import shape_inference
import matplotlib.pyplot as plt

print(f"ONNX version: {onnx.__version__}")
print(f"Default opset: {defs.onnx_opset_version()}")

<a id='1-what-is-an-onnx-operator'></a>
## 1. What is an ONNX Operator?

An ONNX **operator** (op) is a named primitive computation with a formal interface.
Every operator is uniquely identified by a triple:

$$\text{OpID} = (\texttt{domain}, \; \texttt{op\_type}, \; \texttt{opset\_version})$$

- **`domain`** — a namespace string (empty `""` for the default ONNX domain, `"ai.onnx.ml"` for classical ML).
- **`op_type`** — the canonical name: `Conv`, `MatMul`, `Relu`, `Softmax`, etc.
- **`opset_version`** — the operator set version that defines the op's schema (behavior, inputs, attributes).

Each node in an ONNX graph is an instance of exactly one operator:

```
┌──────────────────────────────────────────────────────────────────────┐
│                          NodeProto                                   │
│                                                                      │
│  inputs: ["X", "W", "B"]  ──────┐                                   │
│                                  │     ┌────────────────────┐        │
│  attributes:                     ├────▶│   Operator Kernel  │───▶ outputs: ["Y"]
│    kernel_shape = [3,3]   ───────┤     │      (Conv)        │        │
│    pads = [1,1,1,1]       ───────┤     └────────────────────┘        │
│    strides = [1,1]        ───────┘                                   │
│                                                                      │
│  op_type: "Conv"                                                     │
│  domain:  ""                                                         │
└──────────────────────────────────────────────────────────────────────┘
```

### Inputs vs. Attributes

The distinction between **inputs** and **attributes** is fundamental:

| Property | Inputs (dynamic tensors) | Attributes (static config) |
|----------|--------------------------|----------------------------|
| Binding time | Runtime | Graph construction |
| Source | Computed from other nodes | Embedded in `NodeProto` |
| Examples | `X` (activation), `W` (weight) | `kernel_shape`, `pads`, `axis` |
| Can be overridden? | Yes (feed different data) | No (fixed in the graph) |

A `Conv` node's kernel weights `W` are an **input** (they could be overridden or learned),
but `kernel_shape` is an **attribute** (it defines the operator's structural contract).

In [ ]:
# Inspect the structure of a NodeProto
node = helper.make_node(
    "Conv",
    inputs=["X", "W", "B"],
    outputs=["Y"],
    kernel_shape=[3, 3],
    strides=[1, 1],
    pads=[1, 1, 1, 1],
    dilations=[1, 1],
    group=1,
)

print(f"op_type:    {node.op_type}")
print(f"domain:     {node.domain!r}")
print(f"inputs:     {list(node.input)}")
print(f"outputs:    {list(node.output)}")
print(f"\nAttributes:")
for attr in node.attribute:
    if attr.ints:
        print(f"  {attr.name}: {list(attr.ints)}")
    elif attr.i:
        print(f"  {attr.name}: {attr.i}")
    elif attr.f:
        print(f"  {attr.name}: {attr.f}")

# Look up the schema
schema = defs.get_schema("Conv", defs.onnx_opset_version())
print(f"\nSchema since_version: {schema.since_version}")
print(f"Number of inputs:     {len(schema.inputs)}")
print(f"Number of outputs:    {len(schema.outputs)}")
print(f"Number of attributes: {len(schema.attributes)}")

<a id='2-operator-categories'></a>
## 2. Operator Categories

The ONNX standard operator set (default domain `""`) contains **200+** operators organized
into functional categories. Each category serves a distinct role in the computation graph:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    ONNX STANDARD OPERATOR TAXONOMY                          │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ┌─────────────────┐  ┌─────────────────┐  ┌──────────────────┐           │
│  │  ELEMENTWISE     │  │ LINEAR ALGEBRA   │  │  REDUCTIONS      │           │
│  │  Add, Mul, Exp   │  │ MatMul, Gemm     │  │  ReduceMean/Sum  │           │
│  │  Sqrt, Pow, Clip │  │ Einsum           │  │  ArgMax/Min      │           │
│  └─────────────────┘  └─────────────────┘  └──────────────────┘           │
│                                                                             │
│  ┌─────────────────┐  ┌─────────────────┐  ┌──────────────────┐           │
│  │  NN: CONVOLUTION │  │ NN: ACTIVATION   │  │ NN: NORMALIZATION│           │
│  │  Conv, MaxPool   │  │ Relu, Sigmoid    │  │ BatchNorm        │           │
│  │  ConvTranspose   │  │ Softmax, Gelu    │  │ LayerNorm        │           │
│  └─────────────────┘  └─────────────────┘  └──────────────────┘           │
│                                                                             │
│  ┌─────────────────┐  ┌─────────────────┐  ┌──────────────────┐           │
│  │  TENSOR SHAPE    │  │ SLICING/JOINING  │  │ LOGIC/COMPARISON │           │
│  │  Reshape, Concat │  │ Slice, Gather    │  │ Equal, Where     │           │
│  │  Transpose       │  │ Split, Pad       │  │ Greater, And     │           │
│  └─────────────────┘  └─────────────────┘  └──────────────────┘           │
│                                                                             │
│  ┌─────────────────┐  ┌─────────────────┐  ┌──────────────────┐           │
│  │  QUANTIZATION    │  │ CONTROL FLOW     │  │ RECURRENT (RNN)  │           │
│  │  QuantizeLinear  │  │ If, Loop, Scan   │  │ LSTM, GRU        │           │
│  │  DequantizeLinear│  │ SequenceConstruct│  │ RNN              │           │
│  └─────────────────┘  └─────────────────┘  └──────────────────┘           │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Detailed Category Breakdown

| Category | Count | Examples | Role |
|----------|-------|---------|------|
| Elementwise math | ~25 | Add, Mul, Exp, Sqrt, Clip, Pow | Per-element with broadcasting |
| Linear algebra | ~5 | MatMul, Gemm, Einsum | Matrix/tensor contractions |
| Reduction | ~12 | ReduceMean, ReduceSum, ArgMax | Collapse along axes |
| Tensor shape | ~15 | Reshape, Transpose, Squeeze, Flatten | Change layout without data copy |
| Slicing & joining | ~12 | Slice, Concat, Gather, Split, Pad | Extract/assemble sub-tensors |
| NN: convolution | ~8 | Conv, ConvTranspose, MaxPool, AvgPool | Spatial feature extraction |
| NN: activation | ~17 | Relu, Sigmoid, Softmax, Gelu, Mish | Introduce nonlinearity |
| NN: normalization | ~6 | BatchNorm, LayerNorm, GroupNorm | Stabilize activations |
| NN: recurrent | ~3 | LSTM, GRU, RNN | Sequential modeling |
| Logic & comparison | ~13 | Equal, Greater, Where, And, Or | Masks and conditionals |
| Quantization | ~5 | QuantizeLinear, DequantizeLinear | INT8/FP16 workflows |
| Control flow | ~10 | If, Loop, Scan, SequenceConstruct | Dynamic computation |

In [ ]:
# Enumerate all operators in the default domain
all_schemas = defs.get_all_schemas_with_history()
default_ops = set()
ml_ops = set()
op_versions = {}  # op_name -> list of versions

for schema in all_schemas:
    if schema.domain == '' or schema.domain == 'ai.onnx':
        default_ops.add(schema.name)
        op_versions.setdefault(schema.name, []).append(schema.since_version)
    elif schema.domain == 'ai.onnx.ml':
        ml_ops.add(schema.name)

print(f"Default domain operators:  {len(default_ops)}")
print(f"ai.onnx.ml operators:      {len(ml_ops)}")
print(f"Total unique operators:    {len(default_ops) + len(ml_ops)}")

print(f"\nFirst 40 operators (alphabetical):")
for i, op in enumerate(sorted(default_ops)[:40], 1):
    versions = sorted(op_versions.get(op, []))
    print(f"  {i:>3}. {op:<30s} versions: {versions}")

<a id='3-conv'></a>
## 3. Conv — Mathematical Definition

The `Conv` operator implements $N$-dimensional convolution — the foundational operation for
convolutional neural networks (CNNs). For 2D convolution with all parameters:

### Full Formula

$$Y[n, c_{out}, h, w] = \sum_{c_{in}=0}^{C_{in}/g - 1} \sum_{r=0}^{k_H - 1} \sum_{s=0}^{k_W - 1} X\big[n, \; g_{idx} \cdot \tfrac{C_{in}}{g} + c_{in}, \; h \cdot s_H + r \cdot d_H - p_t, \; w \cdot s_W + s \cdot d_W - p_l\big] \cdot W[c_{out}, c_{in}, r, s] \;+\; b[c_{out}]$$

where $g_{idx} = \lfloor c_{out} / (C_{out}/g) \rfloor$ is the **group index**.

### Output Spatial Dimensions

$$H_{out} = \left\lfloor \frac{H_{in} + p_t + p_b - d_H \cdot (k_H - 1) - 1}{s_H} \right\rfloor + 1$$

$$W_{out} = \left\lfloor \frac{W_{in} + p_l + p_r - d_W \cdot (k_W - 1) - 1}{s_W} \right\rfloor + 1$$

### Sliding Window Visualization

```
  Input X (1 channel, 5×5)        Kernel W (3×3)       Output Y (3×3)
  ┌───┬───┬───┬───┬───┐          ┌───┬───┬───┐        ┌───┬───┬───┐
  │ x │ x │ x │   │   │          │ w │ w │ w │        │   │   │   │
  ├───┼───┼───┼───┼───┤          ├───┼───┼───┤        ├───┼───┼───┤
  │ x │ x │ x │   │   │  ────▶   │ w │ w │ w │  ───▶  │ Y │   │   │
  ├───┼───┼───┼───┼───┤          ├───┼───┼───┤        ├───┼───┼───┤
  │ x │ x │ x │   │   │          │ w │ w │ w │        │   │   │   │
  ├───┼───┼───┼───┼───┤          └───┴───┴───┘        └───┴───┴───┘
  │   │   │   │   │   │
  ├───┼───┼───┼───┼───┤          Y[0,0] = Σ(x·w) + b
  │   │   │   │   │   │
  └───┴───┴───┴───┴───┘

  stride=1, pad=0, dilation=1  →  H_out = (5+0+0-1*(3-1)-1)/1+1 = 3
```

### Grouped and Depthwise Convolution

When `group > 1`, the input and output channels are split into `g` groups.
Each group convolves independently:

- **Standard** ($g=1$): $W \in \mathbb{R}^{C_{out} \times C_{in} \times k_H \times k_W}$, FLOPs $= 2 \cdot N \cdot C_{out} \cdot H_{out} \cdot W_{out} \cdot C_{in} \cdot k_H \cdot k_W$
- **Grouped** ($g > 1$): $W \in \mathbb{R}^{C_{out} \times (C_{in}/g) \times k_H \times k_W}$, FLOPs reduced by factor $g$
- **Depthwise** ($g = C_{in} = C_{out}$): $W \in \mathbb{R}^{C \times 1 \times k_H \times k_W}$, each channel filtered independently

```
  Standard Conv (g=1)          Grouped Conv (g=2)          Depthwise Conv (g=C)
  ┌──────────────┐             ┌──────┐  ┌──────┐          ┌──┐┌──┐┌──┐┌──┐
  │ All C_in     │             │Group1│  │Group2│          │c1││c2││c3││c4│
  │ channels     │──▶ C_out    │C/2   │  │C/2   │          │  ││  ││  ││  │
  │ interact     │             │      │  │      │          │  ││  ││  ││  │
  └──────────────┘             └──────┘  └──────┘          └──┘└──┘└──┘└──┘
  Params: C_out·C_in·k²        Params: C_out·(C_in/g)·k²  Params: C·1·k²
```

### Attribute Reference

| Attribute | Type | Default | Description |
|-----------|------|---------|-------------|
| `kernel_shape` | `int[]` | (required) | Spatial extent: $[k_H, k_W]$ |
| `strides` | `int[]` | `[1,1]` | Step between window positions: $[s_H, s_W]$ |
| `pads` | `int[]` | `[0,0,0,0]` | Padding: $[p_t, p_l, p_b, p_r]$ |
| `dilations` | `int[]` | `[1,1]` | Kernel dilation: $[d_H, d_W]$ |
| `group` | `int` | `1` | Grouped convolution ($g = C_{in}$ → depthwise) |
| `auto_pad` | `string` | `"NOTSET"` | Automatic padding mode |

In [ ]:
# Build and run a Conv + Relu model, verifying output dimensions
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 3, 8, 8])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

np.random.seed(42)
W_init = numpy_helper.from_array(
    np.random.randn(16, 3, 3, 3).astype(np.float32) * 0.1, name="W")
B_init = numpy_helper.from_array(np.zeros(16, dtype=np.float32), name="B")

nodes = [
    helper.make_node("Conv", ["X", "W", "B"], ["conv_out"],
                     kernel_shape=[3, 3], pads=[1, 1, 1, 1], strides=[1, 1]),
    helper.make_node("Relu", ["conv_out"], ["Y"]),
]

graph = helper.make_graph(nodes, "conv_relu", [X], [Y], initializer=[W_init, B_init])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
model = shape_inference.infer_shapes(model)
checker.check_model(model)

# Manually compute output shape using the formula
H_in, W_in = 8, 8
k_H, k_W = 3, 3
s_H, s_W = 1, 1
p_t, p_l, p_b, p_r = 1, 1, 1, 1
d_H, d_W = 1, 1

H_out = (H_in + p_t + p_b - d_H * (k_H - 1) - 1) // s_H + 1
W_out = (W_in + p_l + p_r - d_W * (k_W - 1) - 1) // s_W + 1

print(f"Input shape:  [1, 3, {H_in}, {W_in}]")
print(f"Kernel shape: [16, 3, {k_H}, {k_W}]")
print(f"Output shape: [1, 16, {H_out}, {W_out}]")
print(f"FLOPs:        {2 * 1 * 16 * H_out * W_out * 3 * k_H * k_W:,}")

# Run with ReferenceEvaluator
try:
    from onnx.reference import ReferenceEvaluator
    ev = ReferenceEvaluator(model)
    x = np.random.randn(1, 3, 8, 8).astype(np.float32)
    y = ev.run(None, {"X": x})[0]
    print(f"\nReferenceEvaluator output shape: {y.shape}")
    print(f"All non-negative (Relu applied): {(y >= 0).all()}")
    print(f"Value range: [{y.min():.4f}, {y.max():.4f}]")
except ImportError:
    print("\nReferenceEvaluator not available (onnx >= 1.14 required)")

In [ ]:
# Demonstrate grouped and depthwise convolution
def build_conv_model(C_in, C_out, kernel, group, name):
    """Build a Conv model with specified group parameter."""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, C_in, 8, 8])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
    W = numpy_helper.from_array(
        np.random.randn(C_out, C_in // group, kernel, kernel).astype(np.float32) * 0.1,
        name="W")
    node = helper.make_node("Conv", ["X", "W"], ["Y"],
                            kernel_shape=[kernel, kernel],
                            pads=[kernel//2]*4,
                            group=group)
    graph = helper.make_graph([node], name, [X], [Y], initializer=[W])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    return shape_inference.infer_shapes(model)

configs = [
    (32, 64, 3, 1,  "standard"),
    (32, 64, 3, 2,  "grouped_g2"),
    (32, 64, 3, 4,  "grouped_g4"),
    (32, 32, 3, 32, "depthwise"),
]

print(f"{'Config':<20} {'Group':>5} {'W shape':>20} {'Params':>10} {'FLOPs':>12}")
print("─" * 72)
for c_in, c_out, k, g, name in configs:
    m = build_conv_model(c_in, c_out, k, g, name)
    w_shape = f"[{c_out},{c_in//g},{k},{k}]"
    params = c_out * (c_in // g) * k * k
    flops = 2 * c_out * 8 * 8 * (c_in // g) * k * k
    print(f"{name:<20} {g:>5} {w_shape:>20} {params:>10,} {flops:>12,}")

<a id='4-matmul-and-gemm'></a>
## 4. MatMul and Gemm

### MatMul — General Matrix Multiplication

MatMul performs matrix multiplication with **batch broadcasting** on leading dimensions.
For inputs $A \in \mathbb{R}^{\ldots \times M \times K}$ and $B \in \mathbb{R}^{\ldots \times K \times N}$:

$$C[\ldots, i, j] = \sum_{k=0}^{K-1} A[\ldots, i, k] \cdot B[\ldots, k, j]$$

The output is $C \in \mathbb{R}^{\ldots \times M \times N}$, with leading batch dimensions broadcast.

**Shape rules for MatMul** (distinct from Gemm!):

| A shape | B shape | C shape | Notes |
|---------|---------|---------|-------|
| $[M, K]$ | $[K, N]$ | $[M, N]$ | Standard 2D matmul |
| $[B, M, K]$ | $[B, K, N]$ | $[B, M, N]$ | Batched (same batch) |
| $[1, M, K]$ | $[B, K, N]$ | $[B, M, N]$ | Batch broadcast |
| $[K]$ | $[K, N]$ | $[N]$ | Vector × matrix |
| $[M, K]$ | $[K]$ | $[M]$ | Matrix × vector |

### Gemm (General Matrix Multiply)

Gemm fuses multiplication and bias addition:

$$Y = \alpha \cdot A' B' + \beta \cdot C$$

where $A' = A^T$ if `transA=1` else $A$, and $B' = B^T$ if `transB=1` else $B$.

| Attribute | Type | Default | Description |
|-----------|------|---------|-------------|
| `alpha` | `float` | 1.0 | Scalar multiplier for $AB$ |
| `beta` | `float` | 1.0 | Scalar multiplier for $C$ |
| `transA` | `int` | 0 | Transpose $A$? |
| `transB` | `int` | 0 | Transpose $B$? |

Gemm is commonly produced by **fusing `MatMul + Add`** during graph optimization —
the combined operation maps directly to hardware BLAS routines (sgemm, dgemm).

In [ ]:
# MatMul with batch broadcasting
A_info = helper.make_tensor_value_info("A", TensorProto.FLOAT, [2, 3, 4])
B_info = helper.make_tensor_value_info("B", TensorProto.FLOAT, [2, 4, 5])
C_info = helper.make_tensor_value_info("C", TensorProto.FLOAT, None)

mm_model = helper.make_model(
    helper.make_graph(
        [helper.make_node("MatMul", ["A", "B"], ["C"])],
        "matmul", [A_info, B_info], [C_info]),
    opset_imports=[helper.make_opsetid("", 17)])
mm_model = shape_inference.infer_shapes(mm_model)

try:
    from onnx.reference import ReferenceEvaluator
    ev = ReferenceEvaluator(mm_model)
    a = np.random.randn(2, 3, 4).astype(np.float32)
    b = np.random.randn(2, 4, 5).astype(np.float32)
    c = ev.run(None, {"A": a, "B": b})[0]
    expected = a @ b
    print(f"MatMul: {a.shape} @ {b.shape} → {c.shape}")
    print(f"Matches numpy: {np.allclose(c, expected, atol=1e-6)}")
    print(f"FLOPs: {2 * 2 * 3 * 4 * 5:,}")
except ImportError:
    print("ReferenceEvaluator not available")

# Gemm: Y = alpha * A @ B^T + beta * C
A_g = helper.make_tensor_value_info("A", TensorProto.FLOAT, [4, 3])
B_g = helper.make_tensor_value_info("B", TensorProto.FLOAT, [5, 3])  # will be transposed
C_g = helper.make_tensor_value_info("bias", TensorProto.FLOAT, [5])
Y_g = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

gemm_model = helper.make_model(
    helper.make_graph(
        [helper.make_node("Gemm", ["A", "B", "bias"], ["Y"],
                          alpha=1.0, beta=1.0, transA=0, transB=1)],
        "gemm", [A_g, B_g, C_g], [Y_g]),
    opset_imports=[helper.make_opsetid("", 17)])

try:
    ev = ReferenceEvaluator(gemm_model)
    a = np.random.randn(4, 3).astype(np.float32)
    b = np.random.randn(5, 3).astype(np.float32)
    bias = np.random.randn(5).astype(np.float32)
    y = ev.run(None, {"A": a, "B": b, "bias": bias})[0]
    expected = a @ b.T + bias
    print(f"\nGemm (transB=1): {a.shape} @ {b.shape}^T + bias → {y.shape}")
    print(f"Matches manual: {np.allclose(y, expected, atol=1e-5)}")
except ImportError:
    pass

<a id='5-softmax'></a>
## 5. Softmax — Numerical Stability

### Definition

Softmax converts a vector of real numbers into a probability distribution:

$$\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{K} e^{x_j}}$$

**Properties:**
- Output values $\in (0, 1)$
- Sum to 1: $\sum_i \text{Softmax}(x)_i = 1$
- Monotonic: $x_i > x_j \Rightarrow \text{Softmax}(x)_i > \text{Softmax}(x)_j$
- Translation invariant: $\text{Softmax}(x + c) = \text{Softmax}(x)$ for any scalar $c$

### Numerical Stability — The Max-Subtraction Trick

Direct computation of $e^{x_i}$ can overflow when $x_i$ is large (e.g., $e^{1000} = \infty$
in float32). The standard trick subtracts the maximum:

$$\text{Softmax}(x_i) = \frac{e^{x_i - \max(x)}}{\sum_{j} e^{x_j - \max(x)}}$$

This is mathematically equivalent — the $e^{-\max(x)}$ factor cancels in numerator and
denominator — but prevents overflow since $x_i - \max(x) \leq 0$ always.

### Log-Softmax

For numerical stability in loss computation (cross-entropy), prefer `LogSoftmax`:

$$\log \text{Softmax}(x_i) = x_i - \log \sum_{j} e^{x_j} = x_i - \text{LogSumExp}(x)$$

Computing $\log(\text{Softmax}(x))$ directly loses precision for small probabilities;
`LogSoftmax` avoids this by never materializing the intermediate probability.

### Softmax Axis Change (Opset 13)

Before opset 13, `axis` split the input into 2D: left-of-axis × right-of-axis, then
applied softmax over the right portion. From opset 13 onward, softmax applies along
a **single axis** (matching PyTorch/NumPy behavior).

In [ ]:
# Demonstrate softmax with numerical stability analysis
def manual_softmax_stable(z, axis=-1):
    z_shifted = z - np.max(z, axis=axis, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=axis, keepdims=True)

def manual_softmax_naive(z, axis=-1):
    exp_z = np.exp(z)  # no shift — can overflow!
    return exp_z / np.sum(exp_z, axis=axis, keepdims=True)

# Build ONNX softmax model
logits_info = helper.make_tensor_value_info("logits", TensorProto.FLOAT, [1, 10])
probs_info = helper.make_tensor_value_info("probs", TensorProto.FLOAT, [1, 10])
sm_model = helper.make_model(
    helper.make_graph(
        [helper.make_node("Softmax", ["logits"], ["probs"], axis=1)],
        "softmax", [logits_info], [probs_info]),
    opset_imports=[helper.make_opsetid("", 17)])

try:
    from onnx.reference import ReferenceEvaluator
    ev = ReferenceEvaluator(sm_model)

    # Normal range
    z = np.array([[1., 2., 3., 4., 5., 6., 7., 8., 9., 10.]], dtype=np.float32)
    probs = ev.run(None, {"logits": z})[0]
    manual = manual_softmax_stable(z, axis=1)
    print("Normal range logits [1..10]:")
    print(f"  ONNX output:   {np.round(probs[0], 5)}")
    print(f"  Sum:           {probs.sum():.8f}")
    print(f"  Match manual:  {np.allclose(probs, manual)}")

    # Large values — overflow risk
    z_large = np.array([[100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]], dtype=np.float32)
    probs_large = ev.run(None, {"logits": z_large})[0]
    naive = manual_softmax_naive(z_large, axis=1)
    stable = manual_softmax_stable(z_large, axis=1)
    print(f"\nLarge logits [100..1000]:")
    print(f"  ONNX (stable): sum={probs_large.sum():.6f}, has NaN={np.isnan(probs_large).any()}")
    print(f"  Naive manual:  sum={naive.sum():.6f}, has NaN={np.isnan(naive).any()}")
    print(f"  Stable manual: sum={stable.sum():.6f}, has NaN={np.isnan(stable).any()}")
except ImportError:
    print("ReferenceEvaluator not available")

<a id='6-batchnormalization'></a>
## 6. BatchNormalization

BatchNormalization normalizes activations to zero mean and unit variance, then applies
a learned affine transform:

$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} \cdot \gamma + \beta$$

where:
- $\mu_B = \frac{1}{m} \sum_{i=1}^{m} x_i$ is the batch/running mean
- $\sigma_B^2 = \frac{1}{m} \sum_{i=1}^{m} (x_i - \mu_B)^2$ is the batch/running variance
- $\gamma$ is the learned **scale** parameter
- $\beta$ is the learned **shift** parameter
- $\epsilon$ is a small constant for numerical stability (default $10^{-5}$)

### Training vs. Inference

| Phase | Mean | Variance | Notes |
|-------|------|----------|-------|
| **Training** | Batch mean $\mu_B$ | Batch variance $\sigma_B^2$ | Running stats updated via EMA |
| **Inference** | Running mean $\mu_{\text{run}}$ | Running variance $\sigma_{\text{run}}^2$ | Pre-computed, deterministic |

Running statistics update (during training):

$$\mu_{\text{run}} \leftarrow (1 - \text{momentum}) \cdot \mu_{\text{run}} + \text{momentum} \cdot \mu_B$$
$$\sigma_{\text{run}}^2 \leftarrow (1 - \text{momentum}) \cdot \sigma_{\text{run}}^2 + \text{momentum} \cdot \sigma_B^2$$

In ONNX models (inference mode), the pre-computed running statistics are used directly.

### Input Specification

| Position | Name | Shape | Description |
|----------|------|-------|-------------|
| 0 | X | $[N, C, \ldots]$ | Input tensor (batch × channels × spatial) |
| 1 | scale | $[C]$ | Learned $\gamma$ per channel |
| 2 | B | $[C]$ | Learned $\beta$ per channel |
| 3 | input_mean | $[C]$ | Running mean $\mu$ per channel |
| 4 | input_var | $[C]$ | Running variance $\sigma^2$ per channel |

In [ ]:
# Build and verify BatchNormalization with known statistics
C = 4
X_bn = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, C, 4, 4])
Y_bn = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

# Known running stats: mean=2.0, var=4.0, scale=0.5, bias=1.0
scale_init = numpy_helper.from_array(np.full(C, 0.5, dtype=np.float32), "scale")
bias_init = numpy_helper.from_array(np.full(C, 1.0, dtype=np.float32), "bias")
mean_init = numpy_helper.from_array(np.full(C, 2.0, dtype=np.float32), "mean")
var_init = numpy_helper.from_array(np.full(C, 4.0, dtype=np.float32), "var")

bn_model = helper.make_model(
    helper.make_graph(
        [helper.make_node("BatchNormalization",
                          ["X", "scale", "bias", "mean", "var"], ["Y"],
                          epsilon=1e-5)],
        "bn", [X_bn], [Y_bn],
        initializer=[scale_init, bias_init, mean_init, var_init]),
    opset_imports=[helper.make_opsetid("", 17)])
checker.check_model(bn_model)

try:
    from onnx.reference import ReferenceEvaluator
    ev = ReferenceEvaluator(bn_model)
    x = np.random.randn(2, C, 4, 4).astype(np.float32) * 3 + 5
    y = ev.run(None, {"X": x})[0]

    # Manual computation: y = (x - mean) / sqrt(var + eps) * scale + bias
    eps = 1e-5
    expected = (x - 2.0) / np.sqrt(4.0 + eps) * 0.5 + 1.0

    print(f"Input stats:    mean={x.mean():.4f}, std={x.std():.4f}")
    print(f"Output stats:   mean={y.mean():.4f}, std={y.std():.4f}")
    print(f"Expected stats: mean={expected.mean():.4f}, std={expected.std():.4f}")
    print(f"Match expected: {np.allclose(y, expected, atol=1e-5)}")
    print(f"Max abs error:  {np.max(np.abs(y - expected)):.2e}")
except ImportError:
    print("ReferenceEvaluator not available")

<a id='7-activation-functions'></a>
## 7. Activation Functions

Activation functions introduce **nonlinearity** into the network. Without them, any
stack of linear layers collapses to a single linear transformation:

$$f(W_2 \cdot W_1 \cdot x + b) \neq W_2 \cdot W_1 \cdot x + b \quad \text{(when } f \text{ is nonlinear)}$$

### Mathematical Definitions

| Activation | Formula | Range | Properties |
|-----------|---------|-------|------------|
| **Relu** | $f(x) = \max(0, x)$ | $[0, \infty)$ | Sparse, fast, dying neuron risk |
| **Sigmoid** | $\sigma(x) = \frac{1}{1 + e^{-x}}$ | $(0, 1)$ | Smooth, saturates, vanishing gradient |
| **Tanh** | $f(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$ | $(-1, 1)$ | Zero-centered, still saturates |
| **Gelu** | $f(x) = x \cdot \Phi(x)$ | $\approx(-0.17, \infty)$ | Smooth approximation of Relu |
| **LeakyRelu** | $f(x) = \max(\alpha x, x)$ | $(-\infty, \infty)$ | No dying neurons |
| **Elu** | $f(x) = \begin{cases} x & x > 0 \\ \alpha(e^x - 1) & x \leq 0 \end{cases}$ | $(-\alpha, \infty)$ | Smooth for $x < 0$ |
| **Selu** | $f(x) = \lambda \begin{cases} x & x > 0 \\ \alpha(e^x - 1) & x \leq 0 \end{cases}$ | self-normalizing | Fixed $\alpha, \lambda$ |
| **HardSigmoid** | $f(x) = \max(0, \min(1, \alpha x + \beta))$ | $[0, 1]$ | Piecewise linear approx |

where $\Phi(x) = \frac{1}{2}[1 + \text{erf}(x / \sqrt{2})]$ is the CDF of $\mathcal{N}(0, 1)$.

### Derivatives (for gradient flow understanding)

| Activation | Derivative | Gradient issue |
|-----------|-----------|----------------|
| Relu | $f'(x) = \begin{cases} 1 & x > 0 \\ 0 & x \leq 0 \end{cases}$ | Dead neurons when $x \leq 0$ |
| Sigmoid | $\sigma'(x) = \sigma(x)(1 - \sigma(x))$ | Max gradient $= 0.25$ → vanishing |
| Tanh | $f'(x) = 1 - \tanh^2(x)$ | Max gradient $= 1.0$ → still vanishing |
| Gelu | $f'(x) = \Phi(x) + x \cdot \phi(x)$ | Smooth, well-behaved |

In [ ]:
# Visualize all activation functions and their derivatives
x_vals = np.linspace(-5, 5, 500)

from math import erf as _erf
_verf = np.vectorize(_erf)

activations = {
    'Relu': np.maximum(0, x_vals),
    'Sigmoid': 1.0 / (1.0 + np.exp(-x_vals)),
    'Tanh': np.tanh(x_vals),
    'Gelu': x_vals * 0.5 * (1 + _verf(x_vals / np.sqrt(2))),
    'LeakyRelu (a=0.1)': np.where(x_vals > 0, x_vals, 0.1 * x_vals),
    'Elu (a=1)': np.where(x_vals > 0, x_vals, np.exp(x_vals) - 1),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']

for ax, (name, y_vals), color in zip(axes.flat, activations.items(), colors):
    ax.plot(x_vals, y_vals, color=color, linewidth=2.5)
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlim(-5, 5)
    ax.grid(True, alpha=0.2)
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')

plt.suptitle('ONNX Activation Functions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<a id='8-reduction-operators'></a>
## 8. Reduction Operators

Reduction operators collapse one or more axes of a tensor, computing aggregate statistics.

### Formal Definition

For input $X \in \mathbb{R}^{d_1 \times d_2 \times \ldots \times d_n}$ and reduction axis $k$:

$$Y_{\text{ReduceSum}}[\ldots, \hat{k}, \ldots] = \sum_{i=0}^{d_k - 1} X[\ldots, i, \ldots]$$

$$Y_{\text{ReduceMean}}[\ldots, \hat{k}, \ldots] = \frac{1}{d_k} \sum_{i=0}^{d_k - 1} X[\ldots, i, \ldots]$$

$$Y_{\text{ReduceMax}}[\ldots, \hat{k}, \ldots] = \max_{i \in [0, d_k)} X[\ldots, i, \ldots]$$

$$Y_{\text{ArgMax}}[\ldots, \hat{k}, \ldots] = \arg\max_{i \in [0, d_k)} X[\ldots, i, \ldots]$$

### Operator Table

| Operator | Formula | Common Use Case |
|----------|---------|------------------|
| `ReduceSum` | $y = \sum_{i \in \text{axis}} x_i$ | Loss aggregation, total activation |
| `ReduceMean` | $y = \frac{1}{n} \sum_{i \in \text{axis}} x_i$ | Layer normalization, pooling |
| `ReduceMax` | $y = \max_{i \in \text{axis}} x_i$ | Max pooling, attention |
| `ReduceMin` | $y = \min_{i \in \text{axis}} x_i$ | Range computation |
| `ReduceProd` | $y = \prod_{i \in \text{axis}} x_i$ | Shape size computation |
| `ReduceL2` | $y = \sqrt{\sum_{i \in \text{axis}} x_i^2}$ | Norm computation |
| `ArgMax` | $y = \arg\max_{i \in \text{axis}} x_i$ | Classification prediction |

### Shape Behavior

- With `keepdims=1`: axis $k$ becomes size 1 → $d_1 \times \ldots \times 1 \times \ldots \times d_n$
- With `keepdims=0`: axis $k$ is removed → $d_1 \times \ldots \times d_{k-1} \times d_{k+1} \times \ldots \times d_n$

In [ ]:
# Demonstrate reduction operators with shape analysis
x = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]], dtype=np.float32)
print(f"Input ({x.shape}):")
print(x)

reductions = [
    ('ReduceSum, axis=0',    np.sum(x, axis=0)),
    ('ReduceSum, axis=1',    np.sum(x, axis=1)),
    ('ReduceMean, axis=1',   np.mean(x, axis=1)),
    ('ReduceMax, axis=0',    np.max(x, axis=0)),
    ('ReduceMin, axis=1',    np.min(x, axis=1)),
    ('ReduceProd, axis=0',   np.prod(x, axis=0)),
    ('ArgMax, axis=1',       np.argmax(x, axis=1)),
    ('ReduceL2, axis=1',     np.sqrt(np.sum(x**2, axis=1))),
    ('ReduceSum, all axes',  np.sum(x)),
]

print(f"\n{'Operation':<25} │ {'Result':>25} │ Shape")
print("─" * 65)
for name, result in reductions:
    print(f"{name:<25} │ {str(result):>25} │ {result.shape}")

<a id='9-tensor-manipulation'></a>
## 9. Tensor Manipulation

Tensor manipulation operators change the **layout** or extract portions of tensors
without modifying the underlying data values.

### Operator Reference

| Operator | Operation | Example | Constraint |
|----------|-----------|---------|------------|
| `Reshape` | Change shape, preserve data | $[2, 6] \to [3, 4]$ | $\prod d_i^{in} = \prod d_j^{out}$ |
| `Transpose` | Permute dimensions | $[N,C,H,W] \to [N,H,W,C]$ | Bijective perm |
| `Squeeze` | Remove size-1 dims | $[1,3,1,4] \to [3,4]$ | Target dim must be 1 |
| `Unsqueeze` | Insert size-1 dims | $[3,4] \to [1,3,4]$ | Axis in valid range |
| `Flatten` | Collapse dims to 2D | $[N,C,H,W] \to [N, C{\cdot}H{\cdot}W]$ | Split at `axis` |
| `Gather` | Index-based lookup | Embedding table | indices must be valid |
| `Concat` | Join along axis | Feature concatenation | All other dims match |
| `Split` | Divide along axis | Multi-head split | Sum of splits = dim |
| `Slice` | Extract sub-tensor | Window extraction | In-bounds |

### Reshape Constraint

Reshape requires the total number of elements to be preserved:

$$\prod_{i=0}^{r_{in}-1} d_i^{\text{input}} = \prod_{j=0}^{r_{out}-1} d_j^{\text{output}}$$

A single dimension may be set to $-1$ ("infer from the total"):

$$d_{-1} = \frac{\prod d_i^{\text{input}}}{\prod_{j \neq -1} d_j^{\text{output}}}$$

In [ ]:
# Demonstrate tensor manipulation operators
x = np.arange(24, dtype=np.float32).reshape(2, 3, 4)
print(f"Original shape: {x.shape}  (24 elements)")

demos = [
    ("Reshape [6, 4]",      x.reshape(6, 4).shape),
    ("Reshape [2, -1]",     x.reshape(2, -1).shape),
    ("Reshape [-1]",        x.reshape(-1).shape),
    ("Flatten (axis=1)",    x.reshape(x.shape[0], -1).shape),
    ("Transpose [0,2,1]",   np.transpose(x, [0, 2, 1]).shape),
    ("Transpose [2,1,0]",   np.transpose(x, [2, 1, 0]).shape),
]
for name, shape in demos:
    print(f"  {name:<25} → {shape}")

# Squeeze / Unsqueeze
y = np.zeros((1, 3, 1, 4))
print(f"\nSqueeze [1,3,1,4] → {np.squeeze(y).shape}")
print(f"Unsqueeze [3,4] axis=0 → {np.expand_dims(np.zeros((3, 4)), 0).shape}")

# Gather (embedding lookup)
embeddings = np.random.randn(100, 64).astype(np.float32)
indices = np.array([3, 7, 42, 99])
gathered = embeddings[indices]
print(f"\nGather: embeddings{embeddings.shape}[{indices}] → {gathered.shape}")

# Build an ONNX model with Transpose + Reshape
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 3, 4, 4])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
shape_val = numpy_helper.from_array(np.array([2, -1], dtype=np.int64), "shape")

manip_model = helper.make_model(
    helper.make_graph(
        [
            helper.make_node("Transpose", ["X"], ["t1"], perm=[0, 2, 3, 1]),
            helper.make_node("Reshape", ["t1", "shape"], ["Y"]),
        ],
        "manip", [X], [Y], initializer=[shape_val]),
    opset_imports=[helper.make_opsetid("", 17)])
manip_model = shape_inference.infer_shapes(manip_model)

print("\nONNX model intermediate shapes:")
for vi in manip_model.graph.value_info:
    dims = [d.dim_param if d.dim_param else d.dim_value
            for d in vi.type.tensor_type.shape.dim]
    print(f"  {vi.name}: {dims}")
for o in manip_model.graph.output:
    dims = [d.dim_param if d.dim_param else d.dim_value
            for d in o.type.tensor_type.shape.dim]
    print(f"  {o.name} (output): {dims}")

<a id='10-broadcasting-rules'></a>
## 10. Broadcasting Rules

Many ONNX operators (Add, Mul, Sub, Div, etc.) support **numpy-style broadcasting**,
which allows elementwise operations on tensors of different shapes.

### Formal Definition

Two dimensions are **compatible** if:

$$\text{compatible}(d_A, d_B) \iff (d_A = d_B) \lor (d_A = 1) \lor (d_B = 1)$$

The **broadcast shape** for each dimension:

$$d_{\text{out},i} = \max(d_{A,i}, d_{B,i})$$

Shapes are right-aligned: if ranks differ, the shorter shape is padded with 1s on the left.

### Broadcasting Algorithm

```
  Step 1: Right-align shapes (pad shorter with leading 1s)
  Step 2: For each dimension pair (d_A, d_B):
          - If d_A == d_B: output dim = d_A
          - If d_A == 1:   output dim = d_B (stretch A)
          - If d_B == 1:   output dim = d_A (stretch B)
          - Otherwise:     ERROR — incompatible shapes

  Example:     A: [8, 1, 6, 1]      B: [7, 1, 5]
  Right-align: A: [8, 1, 6, 1]      B: [1, 7, 1, 5]  (pad B)
  Result:         [8, 7, 6, 5]
                   │  │  │  │
                   8  7  6  5

  Example:     A: [3, 4]            B: [4]  (scalar-like)
  Right-align: A: [3, 4]            B: [1, 4]  (pad B)
  Result:         [3, 4]             ← B stretched along axis 0
```

### Multidirectional vs Unidirectional Broadcasting

| Type | Rule | Used by |
|------|------|---------|
| **Multidirectional** | Both operands can be broadcast | Add, Mul, Sub, Div, etc. |
| **Unidirectional** | Only the second operand is broadcast | Gemm (bias C broadcast to AB shape) |

In [ ]:
# Demonstrate broadcasting with concrete examples
def broadcast_shapes(shape_a, shape_b):
    """Compute broadcast output shape, matching ONNX/numpy rules."""
    rank = max(len(shape_a), len(shape_b))
    a_padded = [1] * (rank - len(shape_a)) + list(shape_a)
    b_padded = [1] * (rank - len(shape_b)) + list(shape_b)
    result = []
    for da, db in zip(a_padded, b_padded):
        if da == db:
            result.append(da)
        elif da == 1:
            result.append(db)
        elif db == 1:
            result.append(da)
        else:
            raise ValueError(f"Incompatible: {da} vs {db}")
    return tuple(result)

test_cases = [
    ((3, 4),    (4,)),           # bias addition
    ((8, 1, 6, 1), (7, 1, 5)),  # classic numpy example
    ((5, 1),    (1, 6)),         # outer product style
    ((1,),      (3, 4, 5)),     # scalar broadcast
    ((3, 1, 5), (3, 4, 5)),     # middle dim broadcast
    ((1, 1, 1), (8, 16, 32)),   # full broadcast
]

print(f"{'Shape A':>20} {'Shape B':>20} │ {'Broadcast Result':>20}")
print("─" * 65)
for sa, sb in test_cases:
    result = broadcast_shapes(sa, sb)
    print(f"{str(sa):>20} {str(sb):>20} │ {str(result):>20}")

# Verify with ONNX Add model
try:
    from onnx.reference import ReferenceEvaluator
    A = helper.make_tensor_value_info("A", TensorProto.FLOAT, [3, 4])
    B = helper.make_tensor_value_info("B", TensorProto.FLOAT, [4])
    C = helper.make_tensor_value_info("C", TensorProto.FLOAT, None)
    add_model = helper.make_model(
        helper.make_graph(
            [helper.make_node("Add", ["A", "B"], ["C"])],
            "add", [A, B], [C]),
        opset_imports=[helper.make_opsetid("", 17)])

    ev = ReferenceEvaluator(add_model)
    a = np.ones((3, 4), dtype=np.float32)
    b = np.array([10, 20, 30, 40], dtype=np.float32)
    c = ev.run(None, {"A": a, "B": b})[0]
    print(f"\nBroadcast Add: {a.shape} + {b.shape} → {c.shape}")
    print(f"Row 0: {c[0]} (bias [10,20,30,40] added to each row)")
except ImportError:
    pass

<a id='11-building-complete-models'></a>
## 11. Building Complete Models

Let's compose operators to build a complete CNN classifier, demonstrating how
individual ops combine into a working model.

**Architecture:** `Conv → BN → ReLU → MaxPool → Flatten → MatMul → Add → Softmax`

```
  Input [N,1,28,28]
      │
      ▼
  ┌──────────┐
  │   Conv   │  [N,1,28,28] → [N,8,28,28]    (3×3 kernel, pad=1)
  └──────────┘
      │
      ▼
  ┌──────────┐
  │ BatchNorm│  [N,8,28,28] → [N,8,28,28]    (normalize per channel)
  └──────────┘
      │
      ▼
  ┌──────────┐
  │   Relu   │  [N,8,28,28] → [N,8,28,28]    (clamp negatives)
  └──────────┘
      │
      ▼
  ┌──────────┐
  │ MaxPool  │  [N,8,28,28] → [N,8,14,14]    (2×2 pool, stride=2)
  └──────────┘
      │
      ▼
  ┌──────────┐
  │ Flatten  │  [N,8,14,14] → [N,1568]        (flatten spatial)
  └──────────┘
      │
      ▼
  ┌──────────┐
  │  MatMul  │  [N,1568] × [1568,10] → [N,10] (fully connected)
  └──────────┘
      │
      ▼
  ┌──────────┐
  │   Add    │  [N,10] + [10] → [N,10]        (add bias)
  └──────────┘
      │
      ▼
  ┌──────────┐
  │ Softmax  │  [N,10] → [N,10]               (probabilities)
  └──────────┘
      │
      ▼
  Output [N,10]  (class probabilities)
```

In [ ]:
# Build a complete CNN classifier from individual operators
np.random.seed(0)
C_in, C_out_conv, H, W = 1, 8, 28, 28
num_classes = 10
pool_size = 2
H_pool, W_pool = H // pool_size, W // pool_size
flat_dim = C_out_conv * H_pool * W_pool

# Graph I/O
X_in = helper.make_tensor_value_info("input", TensorProto.FLOAT, ["batch", C_in, H, W])
Y_out = helper.make_tensor_value_info("probs", TensorProto.FLOAT, ["batch", num_classes])

# Initializers (pretend these are trained weights)
inits = [
    numpy_helper.from_array(np.random.randn(C_out_conv, C_in, 3, 3).astype(np.float32)*0.1, "conv.w"),
    numpy_helper.from_array(np.zeros(C_out_conv, dtype=np.float32), "conv.b"),
    numpy_helper.from_array(np.ones(C_out_conv, dtype=np.float32), "bn.scale"),
    numpy_helper.from_array(np.zeros(C_out_conv, dtype=np.float32), "bn.bias"),
    numpy_helper.from_array(np.zeros(C_out_conv, dtype=np.float32), "bn.mean"),
    numpy_helper.from_array(np.ones(C_out_conv, dtype=np.float32), "bn.var"),
    numpy_helper.from_array(np.random.randn(flat_dim, num_classes).astype(np.float32)*0.01, "fc.w"),
    numpy_helper.from_array(np.zeros(num_classes, dtype=np.float32), "fc.b"),
]

# Nodes in topological order
nodes = [
    helper.make_node("Conv", ["input", "conv.w", "conv.b"], ["conv_out"],
                     kernel_shape=[3,3], pads=[1,1,1,1]),
    helper.make_node("BatchNormalization",
                     ["conv_out", "bn.scale", "bn.bias", "bn.mean", "bn.var"],
                     ["bn_out"], epsilon=1e-5),
    helper.make_node("Relu", ["bn_out"], ["relu_out"]),
    helper.make_node("MaxPool", ["relu_out"], ["pool_out"],
                     kernel_shape=[2,2], strides=[2,2]),
    helper.make_node("Flatten", ["pool_out"], ["flat_out"], axis=1),
    helper.make_node("MatMul", ["flat_out", "fc.w"], ["mm_out"]),
    helper.make_node("Add", ["mm_out", "fc.b"], ["logits"]),
    helper.make_node("Softmax", ["logits"], ["probs"], axis=1),
]

graph = helper.make_graph(nodes, "cnn_classifier", [X_in], [Y_out], initializer=inits)
cnn = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
cnn = shape_inference.infer_shapes(cnn)
checker.check_model(cnn)

total_params = sum(int(np.prod(list(i.dims))) for i in cnn.graph.initializer
                   if i.data_type == TensorProto.FLOAT)
print(f"Model summary:")
print(f"  Operators:  {len(cnn.graph.node)} nodes")
print(f"  Parameters: {total_params:,}")
print(f"  Size:       {len(cnn.SerializeToString()):,} bytes")
print(f"  Op chain:   {' → '.join(n.op_type for n in cnn.graph.node)}")

# Run inference
try:
    from onnx.reference import ReferenceEvaluator
    ev = ReferenceEvaluator(cnn)
    x = np.random.randn(4, 1, 28, 28).astype(np.float32)
    probs = ev.run(None, {"input": x})[0]
    print(f"\nInference:")
    print(f"  Input:  {x.shape}")
    print(f"  Output: {probs.shape}")
    print(f"  Sum of probs (row 0): {probs[0].sum():.6f}")
    print(f"  Predicted classes: {probs.argmax(axis=1)}")
    print(f"  Max probabilities: {probs.max(axis=1).round(4)}")
except Exception as e:
    print(f"  Eval error: {e}")

<a id='12-attention-mechanism-pattern'></a>
## 12. Attention Mechanism Pattern

The **scaled dot-product attention** mechanism — the core of Transformer models — is
implemented using standard ONNX operators:

$$\text{Attention}(Q, K, V) = \text{Softmax}\!\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

where:
- $Q \in \mathbb{R}^{N \times d_k}$ — queries
- $K \in \mathbb{R}^{M \times d_k}$ — keys
- $V \in \mathbb{R}^{M \times d_v}$ — values
- $d_k$ — key dimension (scaling factor)

### ONNX Operator Decomposition

```
          Q              K^T               V
     [N, d_k]        [d_k, M]          [M, d_v]
         │               │                 │
         └───────┬───────┘                 │
                 │                         │
            ┌────▼─────┐                   │
            │  MatMul   │  Q @ K^T         │
            │ [N, M]    │                   │
            └────┬──────┘                   │
                 │                         │
            ┌────▼──────┐                   │
            │   Div     │  / sqrt(d_k)     │
            │ [N, M]    │                   │
            └────┬──────┘                   │
                 │                         │
            ┌────▼──────┐                   │
            │  Softmax  │  row-wise         │
            │ [N, M]    │                   │
            └────┬──────┘                   │
                 │                         │
                 └───────────┬──────────────┘
                             │
                        ┌────▼─────┐
                        │  MatMul   │  attn_weights @ V
                        │ [N, d_v]  │
                        └──────────┘
```

![Attention Pattern](assets/dot_att.png)

In [ ]:
# Build scaled dot-product attention from ONNX operators
batch, num_heads, seq_len, d_k, d_v = 1, 4, 8, 16, 16

# Input specifications (batched multi-head)
Q_info = helper.make_tensor_value_info("Q", TensorProto.FLOAT,
                                        [batch, num_heads, seq_len, d_k])
K_info = helper.make_tensor_value_info("K", TensorProto.FLOAT,
                                        [batch, num_heads, seq_len, d_k])
V_info = helper.make_tensor_value_info("V", TensorProto.FLOAT,
                                        [batch, num_heads, seq_len, d_v])
out_info = helper.make_tensor_value_info("attn_out", TensorProto.FLOAT, None)

# Scale factor: sqrt(d_k)
scale = numpy_helper.from_array(
    np.array(np.sqrt(d_k), dtype=np.float32), "scale")

nodes = [
    # Step 1: Transpose K → K^T (swap last two dims)
    helper.make_node("Transpose", ["K"], ["K_T"],
                     perm=[0, 1, 3, 2]),
    # Step 2: Q @ K^T → attention scores
    helper.make_node("MatMul", ["Q", "K_T"], ["scores"]),
    # Step 3: Scale by 1/sqrt(d_k)
    helper.make_node("Div", ["scores", "scale"], ["scaled_scores"]),
    # Step 4: Softmax along key dimension
    helper.make_node("Softmax", ["scaled_scores"], ["attn_weights"], axis=-1),
    # Step 5: Weighted sum of values
    helper.make_node("MatMul", ["attn_weights", "V"], ["attn_out"]),
]

attn_graph = helper.make_graph(
    nodes, "scaled_dot_product_attention",
    [Q_info, K_info, V_info], [out_info],
    initializer=[scale])
attn_model = helper.make_model(
    attn_graph, opset_imports=[helper.make_opsetid("", 17)])
attn_model = shape_inference.infer_shapes(attn_model)
checker.check_model(attn_model)

print(f"Attention model: {len(attn_model.graph.node)} ops")
print(f"Ops: {[n.op_type for n in attn_model.graph.node]}")

try:
    from onnx.reference import ReferenceEvaluator
    ev = ReferenceEvaluator(attn_model)
    q = np.random.randn(batch, num_heads, seq_len, d_k).astype(np.float32)
    k = np.random.randn(batch, num_heads, seq_len, d_k).astype(np.float32)
    v = np.random.randn(batch, num_heads, seq_len, d_v).astype(np.float32)
    out = ev.run(None, {"Q": q, "K": k, "V": v})[0]

    # Manual verification
    scores_manual = q @ k.transpose(0, 1, 3, 2) / np.sqrt(d_k)
    weights_manual = np.exp(scores_manual - scores_manual.max(axis=-1, keepdims=True))
    weights_manual = weights_manual / weights_manual.sum(axis=-1, keepdims=True)
    out_manual = weights_manual @ v

    print(f"\nInput shapes:  Q={q.shape}, K={k.shape}, V={v.shape}")
    print(f"Output shape:  {out.shape}")
    print(f"Matches manual: {np.allclose(out, out_manual, atol=1e-5)}")
    print(f"Attention weights sum per row: {weights_manual[0, 0, 0].sum():.6f}")
except ImportError:
    print("ReferenceEvaluator not available")

<a id='13-operator-catalog-visualization'></a>
## 13. Operator Catalog Visualization

Let's visualize the full operator catalog — counting operators by category and
analyzing version history.

In [ ]:
# Categorize ONNX operators for visualization
categories = {
    'Math/Elementwise': ['Add', 'Sub', 'Mul', 'Div', 'Neg', 'Abs', 'Exp', 'Log', 'Sqrt',
                         'Pow', 'Ceil', 'Floor', 'Clip', 'Sign', 'Reciprocal', 'Mod',
                         'BitShift', 'BitwiseAnd', 'BitwiseOr', 'BitwiseXor', 'BitwiseNot'],
    'Linear Algebra': ['MatMul', 'Gemm', 'Einsum', 'MatMulInteger', 'QLinearMatMul'],
    'Activation': ['Relu', 'Sigmoid', 'Tanh', 'Softmax', 'LogSoftmax', 'LeakyRelu',
                   'Elu', 'Selu', 'HardSigmoid', 'HardSwish', 'Mish', 'Celu', 'Gelu',
                   'Softplus', 'Softsign', 'ThresholdedRelu', 'PRelu'],
    'Conv/Pool': ['Conv', 'ConvTranspose', 'MaxPool', 'AveragePool',
                  'GlobalAveragePool', 'GlobalMaxPool', 'LpPool', 'MaxRoiPool',
                  'GlobalLpPool', 'QLinearConv', 'ConvInteger', 'MaxUnpool', 'DeformConv'],
    'Normalization': ['BatchNormalization', 'InstanceNormalization',
                      'LayerNormalization', 'GroupNormalization', 'LpNormalization',
                      'MeanVarianceNormalization'],
    'Shape/Layout': ['Reshape', 'Transpose', 'Squeeze', 'Unsqueeze', 'Flatten',
                     'Expand', 'Tile', 'Shape', 'Size', 'DepthToSpace',
                     'SpaceToDepth', 'Identity'],
    'Reduction': ['ReduceMean', 'ReduceSum', 'ReduceMax', 'ReduceMin',
                  'ReduceProd', 'ReduceL1', 'ReduceL2', 'ReduceLogSum',
                  'ReduceLogSumExp', 'ReduceSumSquare', 'ArgMax', 'ArgMin'],
    'Slice/Join': ['Slice', 'Concat', 'Split', 'Gather', 'GatherElements',
                   'GatherND', 'ScatterElements', 'ScatterND', 'Pad',
                   'Compress', 'Resize', 'Upsample'],
    'Logic/Comparison': ['Equal', 'Greater', 'Less', 'GreaterOrEqual', 'LessOrEqual',
                         'And', 'Or', 'Not', 'Xor', 'Where', 'IsNaN', 'IsInf', 'NonZero'],
    'Quantization': ['QuantizeLinear', 'DequantizeLinear', 'DynamicQuantizeLinear'],
    'Control Flow': ['If', 'Loop', 'Scan', 'SequenceConstruct', 'SequenceAt',
                     'SequenceEmpty', 'SequenceInsert', 'SequenceErase',
                     'SequenceLength', 'ConcatFromSequence', 'SplitToSequence'],
    'Other': ['Cast', 'CastLike', 'Constant', 'ConstantOfShape', 'Range',
              'OneHot', 'EyeLike', 'RandomNormal', 'RandomUniform',
              'Bernoulli', 'Dropout', 'Trilu'],
}

cat_counts = {cat: len(ops) for cat, ops in categories.items()}
total_categorized = sum(cat_counts.values())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sorted_cats = sorted(cat_counts.items(), key=lambda x: -x[1])
names = [c[0] for c in sorted_cats]
counts = [c[1] for c in sorted_cats]
colors = plt.cm.Set3(np.linspace(0, 1, len(names)))

bars = axes[0].barh(names, counts, color=colors, edgecolor='white')
axes[0].set_xlabel('Number of Operators')
axes[0].set_title(f'ONNX Operators by Category ({total_categorized} cataloged)',
                  fontweight='bold')
axes[0].invert_yaxis()
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2.,
                str(count), va='center', fontsize=10)

axes[1].pie(counts, labels=names, autopct='%1.0f%%', colors=colors,
            startangle=90, pctdistance=0.82, textprops={'fontsize': 8})
axes[1].set_title('Category Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Analyze operator version history
since_versions = {}
for schema in defs.get_all_schemas_with_history():
    if schema.domain == '' or schema.domain == 'ai.onnx':
        since_versions.setdefault(schema.since_version, set()).add(schema.name)

versions_sorted = sorted(since_versions.keys())
new_per_version = [len(since_versions.get(v, set())) for v in versions_sorted]
cumulative = np.cumsum(new_per_version)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(versions_sorted, new_per_version, color='steelblue', edgecolor='navy', alpha=0.8)
ax1.set_xlabel('Opset Version (since_version)')
ax1.set_ylabel('New/Updated Operator Schemas')
ax1.set_title('Operator Introductions by Opset Version', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

ax2.plot(versions_sorted, cumulative, 'g-o', markersize=5, linewidth=2)
ax2.fill_between(versions_sorted, cumulative, alpha=0.15, color='green')
ax2.set_xlabel('Opset Version')
ax2.set_ylabel('Cumulative Operator Schemas')
ax2.set_title('Cumulative Schema Count Over Opset Versions', fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nVersion history summary:")
for v in versions_sorted:
    ops = sorted(since_versions[v])
    preview = ', '.join(ops[:5])
    suffix = f', ... (+{len(ops)-5} more)' if len(ops) > 5 else ''
    print(f"  Opset {v:>2}: {len(ops):>3} schemas — {preview}{suffix}")

<a id='14-key-takeaways'></a>
## 14. Key Takeaways

1. **ONNX operators** are named primitive computations identified by the triple
   $(\texttt{domain}, \texttt{op\_type}, \texttt{opset\_version})$. Each node in a graph
   consumes dynamic **inputs** (tensors) and static **attributes** (configuration).

2. **Conv2D** computes:
   $$Y[n,c,h,w] = \sum_{k,r,s} X[\ldots] \cdot W[c,k,r,s] + b[c]$$
   with output size governed by kernel, stride, padding, and dilation.
   Grouped convolution ($g > 1$) and depthwise ($g = C$) reduce parameters and FLOPs.

3. **Softmax** converts logits to probabilities:
   $\text{Softmax}(x_i) = e^{x_i - \max(x)} / \sum_j e^{x_j - \max(x)}$
   using the max-subtraction trick for numerical stability.

4. **BatchNorm** normalizes and re-scales:
   $\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$
   using running statistics at inference time.

5. **Broadcasting** enables elementwise operations on differently-shaped tensors:
   dimensions are compatible if equal or one is 1, shapes are right-aligned.

6. **200+ standard operators** span math, linear algebra, activations, convolutions,
   normalizations, reductions, tensor manipulation, logic, quantization, and control flow.

7. Complex patterns like **multi-head attention** are composed from primitive ops:
   $\text{Attention}(Q,K,V) = \text{Softmax}(QK^T / \sqrt{d_k}) \cdot V$

8. Operators compose into complete models by connecting outputs to inputs through
   **named tensors** in a directed acyclic graph (DAG) structure.